In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load CSV
df = pd.read_csv("crawler_time_log.csv")

# ---- Averages ----
avg_times = df.mean()

avg_fetch = avg_times["fetch_time"]
avg_save = avg_times["page_save_time"]
avg_parse = avg_times["page_parse_time"]
avg_score = avg_times["links_score_time"]

# Average link score time per link
df["score_per_link"] = df["links_score_time"] / df["link_count"]
avg_score_per_link = df["score_per_link"].mean()

print("Average times:")
print(f"Fetch: {avg_fetch:.4f}")
print(f"Save: {avg_save:.4f}")
print(f"Parse: {avg_parse:.4f}")
print(f"Link score: {avg_score:.4f}")
print(f"Link score per link: {avg_score_per_link:.6f}")

# ---- Stacked (layer) plot ----
x = range(len(df))

plt.figure()

# Softer color palette (muted)
colors = ["#2d8fda84", "#41d85578", "#f8871578", "#f01c1c84"]

# Stackplot with transparency
plt.stackplot(
    x,
    df["fetch_time"],
    df["page_save_time"],
    df["page_parse_time"],
    df["links_score_time"],
    labels=[
        "URL Fetch",
        "Database Save",
        "Page Parse",
        "Scoring Links"
    ],
    colors=colors,
    alpha=1.0,
    linewidth=0
)

# ---- Average TOTAL processing time ----
df["total_time"] = (
    df["fetch_time"]
    + df["page_save_time"]
    + df["page_parse_time"]
    + df["links_score_time"]
)

avg_total_time = df["total_time"].mean()

plt.axhline(
    avg_total_time,
    color="#23836b",
    linestyle='-.',
    linewidth=2,
    label=f"Avg total ({avg_total_time:.2f}s)",
    alpha= 0.8
)

# ---- Robots.txt delay line ----
plt.axhline(
    2.0,
    color='black',
    linestyle='--',
    linewidth=2,
    label="robots.txt delay (2.0s)",
    alpha=0.7
)

# ---- Remove empty space ----
plt.xlim(0, len(df) - 1)
plt.margins(x=0)
plt.ylim(0, 10)

# Labels
plt.legend(loc="upper left")
plt.xlabel("Crawled Page Number")
plt.ylabel("Time (seconds)")
plt.title("Crawler Page Processing Times")

plt.savefig("crawler_runtimes.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pickle
import numpy as np

with open('final_display/umap.pkl', "rb") as f:
    umap_2d_points = pickle.load(f)

with open('final_display/cluster_labels.pkl', "rb") as f:
    cluster_labels = pickle.load(f)

with open('final_display/titles.pkl', "rb") as f:
    titles = pickle.load(f)

with open('final_display/urls.pkl', "rb") as f:
    urls = pickle.load(f)

with open('final_display/ids.pkl', "rb") as f:
    ids = pickle.load(f)

with open('final_display/cluster_keywords.pkl', "rb") as f:
    cluster_keywords = pickle.load(f)



def compute_metrics(cluster_labels, tp_clusters):
    cluster_labels = np.array(cluster_labels)

    pred_positive = np.isin(cluster_labels, tp_clusters)

    total = len(cluster_labels)
    tp = np.sum(pred_positive)
    fp = 0
    fn = total - tp
    tn = 0

    accuracy = tp / total
    precision = 1.0 if tp > 0 else 0.0
    recall = tp / total
    f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0

    return {
        "total_points": total,
        "tp_points": int(tp),
        "tp_ratio": tp / total,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

compute_metrics(cluster_labels, [1])
